[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/registration/registration.ipynb)

# Point-Cloud Registration

Point-cloud registration finds the rigid motion that carries one set of points onto another, given which point goes where: a scan of an object taken from two places, or a part located against its model. The two sets differ by measurement noise, so the best motion is the one that brings them closest.

A rigid motion is a motor, and a motor moves a point by its sandwich, `motor >> point`. Multiplied out on the right, the sandwich becomes an equation linear in the motor, `target * Motor - Motor * source`, a map on motors that vanishes on the true motion, so the whole motion, turn and shift together, is the flattest direction of one quadratic form on motors. No step centres the clouds or picks an origin: measured against the motor's own metric, which has no unit for a shift, the eigensolver finds for each turn the shift that fits best, which is what centring does by hand. Moving the whole scene moves the fit with it, and the same lines run on the three-sphere.

In the matrix notation of registration the rotation reads as the Kabsch algorithm, a singular value decomposition of the cross-covariance $\sum_k \mathbf y_k \mathbf x_k^{\mathsf T}$ of the two clouds centred on their means, followed by the translation $\bar{\mathbf y} - R\bar{\mathbf x}$ between the means; in the quaternion notation of Horn the same rotation is an eigenvector problem on unit quaternions after the same centring.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Extensor, NumpyContext, stack
from numga.algebras import PGA3D as ga
from examples import instantiate
from examples.util.animation import save_figure
from examples.geometry.registration import render, scenarios

np.set_printoptions(precision=4, suppress=True)

core = instantiate("examples.geometry.registration.core", ga)
cloud, jitter = core.cloud, core.jitter
mv = NumpyContext(ga).multivector

## 1. Noisy correspondences

The source is a cloud of points spread unevenly along x, y and z. The target is the source moved by a known motion, a shift after a turn, with each point then perturbed by a small random motor.

In [ ]:
# The source's points, their spreads along x, y and z and the seed of the noise; the true motion's shift and
# turn; and the noise that perturbs each target point.
n_points, cloud_spread, seed = 60, np.array([2.0, 1.0, 0.5]), 0
shift, turn = np.array([1.5, -0.5, 2.0]), np.array([0.8, -0.6, 1.4])
noise = 0.02

In [ ]:
rng = np.random.default_rng(seed)
source = cloud(n_points, cloud_spread, rng)                                    # [points] Point-
translation = ((stack((mv.xw, mv.yw, mv.zw)) * shift).sum(axis=0) / 2).exp()  # [] Motor+
rotation = ((stack((mv.xy, mv.yz, mv.zx)) * turn).sum(axis=0) / 2).exp()      # [] Motor+
true_motor = translation * rotation                                            # [] Motor+
target = jitter(true_motor >> source, noise, rng)                              # [points] Point-

## 2. One equation for the whole motion

A motor that carries each source point onto its target satisfies `target * motor == motor * source`, linear in the motor. The squared size of that residual, summed over the points, is a form on motors, and the fitted motor is its flattest direction. In flat space the bulk of the residual misses its ideal part, so the form adds the bulk of its complement, and so measures every coefficient. In the figure, the source cloud is drawn in grey, the targets are blue shells, and the aligned points are red, each tied to its target.

In [ ]:
def fit_motor(source: Extensor, target: Extensor) -> Extensor:
    """The fitted motor is the flattest direction of the summed squared residual of the one-sided equations, in whichever algebra the points live."""
    Motor = source.algebra.gatype.rotor()                                      # Rotor+
    source, target = source.normalized(), target.normalized()                  # [points] Point-, Point-
    # target * Motor == Motor * source is linear in the motor: the residual is a map on motors.
    residual = target * Motor - Motor * source                                 # [points] Antivector- <- Motor+

    # Where the metric is degenerate on ideal blades, the bulk norm alone would discard those planes.
    # Adding the weight norm, the bulk norm of the complement, sums the squares of every coefficient.
    bulk = residual.scalar_norm_squared()                                      # [points] Scalar+ <- (Motor+, Motor+)
    weight = residual.dual().scalar_norm_squared()                             # [points] Scalar+ <- (Motor+, Motor+)
    misfit = (bulk + weight).sum(axis=0)                                       # [] Scalar+ <- (Motor+, Motor+)
    # Against the motor's own metric, which in flat space has no unit for a shift, eigh picks for each turn
    # the shift that fits best: centring, without an origin.
    values, motors = misfit.eigh()                                             # [modes] Scalar+, [modes] Motor+
    return motors[values.argmin()].normalized()                                # [] Motor+


fitted = fit_motor(source, target)                                             # [] Motor+
aligned = fitted >> source                                                     # [points] Point-
save_figure(render.draw_registration(source, target, aligned), "registration_one_sided_residual");

## 3. On the three-sphere

Replacing `w0x+y+z+` with `w+x+y+z+` puts the same points and motors on the three-sphere, and the same `fit_motor` registers them without changing a line. The scene is the flat one built in the spherical algebra; the figure draws its points by their x, y and z coordinates.

In [ ]:
sphere = scenarios.SCENES["spherical"]
sphere_source, sphere_target = scenarios.correspondences(sphere)              # [points] Point-, Point-
sphere_fitted = fit_motor(sphere_source, sphere_target)                        # [] Motor+
save_figure(render.draw_registration(sphere_source, sphere_target, sphere_fitted >> sphere_source), "registration_three_sphere");

In [ ]:
# checks
# The fit moves with the scene: the scene carried far off by a motor is fitted by the fit carried along.
distant = ((stack((mv.xw, mv.yw, mv.zw)) * np.array([50.0, 20.0, -30.0])).sum(axis=0) / 2).exp() * (mv.xy * 0.2).exp()
carried_back = distant.inverse() * fit_motor(distant >> source, distant >> target) * distant
assert min(np.abs((carried_back - fitted).kernel).max(), np.abs((carried_back + fitted).kernel).max()) < 1e-9
# Both fits bring the cloud within the noise of the target.
for aligned_points, target_points in ((aligned, target), (sphere_fitted >> sphere_source, sphere_target)):
    rms = (aligned_points.normalized() - target_points.normalized()).dual().norm_squared().mean(axis=0).square_root()
    assert rms < 0.15